# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import pandas as pd

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My baseline rule

I will prioritize pages for refresh using two signals:

- High content age (older pages are more likely to need updating).
- Low CTR (pages getting impressions but fewer clicks may benefit from improved content).

Reason codes:

- OLD_CONTENT
- LOW_CTR
- OLD_CONTENT_AND_LOW_CTR

Action label:

- REFRESH

In [3]:
!git clone https://github.com/karim-yasser/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 164 (delta 70), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.87 MiB | 16.05 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [7]:
import pandas as pd

# Create a copy
baseline = df.copy()

# Initialize score
baseline["score"] = 0
baseline["reason_code"] = ""

# Rule 1: Old content
baseline.loc[baseline["content_age_days"] > 365, "score"] += 1
baseline.loc[baseline["content_age_days"] > 365, "reason_code"] += "OLD_CONTENT "

# Rule 2: Low CTR
baseline.loc[baseline["ctr"] < 2, "score"] += 1
baseline.loc[baseline["ctr"] < 2, "reason_code"] += "LOW_CTR "

# Action label
baseline["action"] = baseline["score"].apply(
    lambda x: "REFRESH" if x >= 1 else "KEEP"
)

baseline.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,5.88,4.55,0.0,good,striking,down,-41.4,1,LOW_CTR,REFRESH
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.00,10.00,0.0,good,page_3_5,down,-57.7,2,OLD_CONTENT LOW_CTR,REFRESH
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,LOW_CTR,REFRESH
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,1.28,3.45,0.0,good,page_1,stable,-13.8,2,OLD_CONTENT LOW_CTR,REFRESH
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,LOW_CTR,REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import os

# Sort by score
ranked = baseline.sort_values(by="score", ascending=False)

# Create output folder if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

ranked[["content_id", "score", "reason_code", "action"]].head(10)


CSV saved successfully!


,content_id,score,reason_code,action
29980,content_81a91fe32bc2,2,OLD_CONTENT LOW_CTR,REFRESH
31,content_24ee79621dbf,2,OLD_CONTENT LOW_CTR,REFRESH
30,content_249298388b45,2,OLD_CONTENT LOW_CTR,REFRESH
27,content_7ea135180dd9,2,OLD_CONTENT LOW_CTR,REFRESH
23,content_2da6ae9d0882,2,OLD_CONTENT LOW_CTR,REFRESH
17,content_761a44afda12,2,OLD_CONTENT LOW_CTR,REFRESH
29967,content_b9e02bd01a73,2,OLD_CONTENT LOW_CTR,REFRESH
29964,content_07ea0872b973,2,OLD_CONTENT LOW_CTR,REFRESH
29963,content_7ba9b154acf6,2,OLD_CONTENT LOW_CTR,REFRESH
29953,content_ac21791a5807,2,OLD_CONTENT LOW_CTR,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

**1. content_81a91fe32bc2**

Action: REFRESH

I selected this page because it is older than one year (460 days) and its CTR is only 0.66%, so it matched both conditions in my rule.

What would make it wrong: If the page was recently updated after this dataset was collected.


**2. content_24ee79621dbf**

Action: REFRESH

This page is very old (480 days) and has a CTR of only 0.10%, which makes it a strong refresh candidate.

What would make it wrong: If the low CTR is only temporary because of seasonal searches.


**3. content_249298388b45**

Action: REFRESH

The page is 438 days old and currently has a CTR of 0%, so it received the highest score.

What would make it wrong: If the page does not receive enough impressions to judge its CTR correctly.


**4. content_7ea135180dd9**

Action: REFRESH

Both of my rule conditions were met, so the page was ranked as a refresh opportunity.

What would make it wrong: If the content has already been improved recently.


**5. content_2da6ae9d0882**

Action: REFRESH

The content is more than one year old and its CTR is still very low.

What would make it wrong: If another performance metric shows the page is actually doing well.


**6. content_761a44afda12**

Action: REFRESH

The page has old content and weak click performance, so the rule gave it a high score.

What would make it wrong: If the CTR has improved since this data was collected.


**7. content_b9e02bd01a73**

Action: REFRESH

This page satisfied both baseline conditions and was ranked near the top.

What would make it wrong: If missing values affected the final score.


**8. content_07ea0872b973**

Action: REFRESH

The page is old and still has a very low CTR, making it a good refresh candidate.

What would make it wrong: If recent optimization already fixed the problem.


**9. content_7ba9b154acf6**

Action: REFRESH

The baseline rule selected this page because it matched both thresholds.

What would make it wrong: If the page belongs to a topic where low CTR is expected.


**10. content_ac21791a5807**

Action: REFRESH

The page received the maximum score because it is old and has poor CTR.

What would make it wrong: If future data shows that its performance is improving.


**11. content_e6cc2aad65ea**

Action: REFRESH

This page has been online for a long time and its CTR is still extremely low.

What would make it wrong: If users are intentionally not clicking because of search intent.


**12. content_fd6261b74d30**

Action: REFRESH

The rule identified this page as needing attention because it satisfies both conditions.

What would make it wrong: If the content was refreshed after the snapshot.


**13. content_a00c249b224d**

Action: REFRESH

The page is old enough to require review and the CTR supports that decision.

What would make it wrong: If there is not enough traffic to evaluate it fairly.


**14. content_a5637401f707**

Action: REFRESH

The page met the age threshold and has almost no click activity.

What would make it wrong: If the keyword naturally receives very few clicks.


**15. content_6de8c512827f**

Action: REFRESH

The baseline score marked this page because both signals point to weak performance.

What would make it wrong: If another engagement metric tells a different story.


**16. content_3d4004b21c6f**

Action: REFRESH

The page follows the same pattern as most of the top-ranked results: old content and low CTR.

What would make it wrong: If the content has already been rewritten.


**17. content_42f36df38d97**

Action: REFRESH

The scoring rule placed this page in the refresh list because both requirements were satisfied.

What would make it wrong: If recent updates are missing from this dataset.


**18. content_d87a116e2c79**

Action: REFRESH

The page is relatively old and has very weak click performance.

What would make it wrong: If the low CTR is caused by temporary search trends.


**19. content_78b97b24e7b9**

Action: REFRESH

This page matched the same refresh rule as the others, so it received a high priority score.

What would make it wrong: If future performance data shows that it is recovering naturally.


**20. content_7ef742c045a0**

Action: REFRESH

The page satisfies both baseline conditions and therefore appears in the top recommendations.

What would make it wrong: If additional signals such as engagement or conversions suggest keeping it unchanged.


### Overall Observation

After reviewing the top 20 pages, I noticed that almost all of them share the same pattern: they are older than one year and have a low CTR. This makes sense because these are the only two signals used in my baseline rule. The rule is simple, easy to explain, and useful as a starting point, but it does not consider engagement, search intent, or recent content updates. In the next stage, a machine learning model should use more features to reduce false positives and produce better recommendations.

In [10]:
ranked[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "content_age_days",
        "ctr"
    ]
].head(20)

,content_id,score,reason_code,action,content_age_days,ctr
29980,content_81a91fe32bc2,2,OLD_CONTENT LOW_CTR,REFRESH,460,0.66
31,content_24ee79621dbf,2,OLD_CONTENT LOW_CTR,REFRESH,480,0.10
30,content_249298388b45,2,OLD_CONTENT LOW_CTR,REFRESH,438,0.00
27,content_7ea135180dd9,2,OLD_CONTENT LOW_CTR,REFRESH,445,0.17
23,content_2da6ae9d0882,2,OLD_CONTENT LOW_CTR,REFRESH,502,0.34
17,content_761a44afda12,2,OLD_CONTENT LOW_CTR,REFRESH,421,0.07
29967,content_b9e02bd01a73,2,OLD_CONTENT LOW_CTR,REFRESH,517,0.08
29964,content_07ea0872b973,2,OLD_CONTENT LOW_CTR,REFRESH,517,0.68
29963,content_7ba9b154acf6,2,OLD_CONTENT LOW_CTR,REFRESH,482,0.00
29953,content_ac21791a5807,2,OLD_CONTENT LOW_CTR,REFRESH,445,0.16


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some pages may be selected only because they are old or have low CTR. The rule ignores content quality, business value, and recent manual updates, so some recommendations may be false positives.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.